# Lab 4 · Classification metrics and CNN design — Independent assignment

**BME · MSc Deep Learning / VITMMA19 · Fall 2026/2027**
Department of Telecommunications and Artificial Intelligence
**Instructor:** Dr. Mohammed Salah Al-Radhi

**60 minutes, including five minutes to check and submit · five tasks · 20 points**

### Challenge: a digit reader that must work wherever the digit appears

A form scanner crops a 32×32 field around each handwritten digit. The training images were collected by an older system that always **centred** the digit. In use, the digit can appear **anywhere** in the field. You will test which ideas from the lecture make a classifier robust to this shift, evaluate the result beyond accuracy, check one claim, and justify one final model.

Each run trains for 15 epochs and takes roughly 5–15 seconds on a CPU. The intellectual work is in the predictions and the explanations, not in waiting for training.

| Task | Minutes | Points |
| --- | ---: | ---: |
| T1 · Debug a classification loss | 8 | 3 |
| T2 · Predict shapes, parameters and receptive fields | 10 | 4 |
| T3 · Does a CNN handle digits it has only seen centred? | 17 | 6 |
| T4 · Evaluate beyond accuracy | 12 | 4 |
| T5 · Test a claim, freeze the decision, test once | 8 | 3 |
| Check and submit | 5 | — |

**Rules.** Write each prediction *before* running the corresponding cell. Use validation data for every decision; the test set is opened once, in T5. Marks reward correct methods and explanations supported by your own outputs, not a particular accuracy. A failed experiment that is well explained can earn full credit. Optional deeper investigations follow T5.

Use the guided notebook, `01_BME_DL_Lab4_Guided.ipynb`, and the official documentation as references. Submit **one executed `.ipynb` file to Moodle by the end of the lab**. Intentional TODOs stop the template until you complete them.

## Your details

**Name:** YOUR NAME  
**Neptun code:** YOUR NEPTUN CODE  
**Assistance declaration:** *None*, or name each tool (e.g. ChatGPT, Claude, Copilot), what you used it for, and how you checked its output. If you used AI, T5a asks you to test one of its suggestions.

## 0. Setup and rules — provided

Save a personal copy in Drive. **A CPU runtime is sufficient; a GPU is optional.** Run the three setup cells first. No download is needed: the digits are packaged with scikit-learn.

**Data.** The same 1,797 scikit-learn digits and stratified split IDs as Lab 3 (**1,077 training / 360 validation / 360 test**), now enlarged to 16×16 and pasted onto an empty 32×32 field. No noise is added.

- **Training:** every digit is centred (top-left corner at row 8, column 8), unless you switch on augmentation.
- **Validation:** two views of the *same* 360 images: *centred*, and *shifted* (each image at one fixed random position). Because the images are identical, the two views form a paired comparison.
- **Test:** the same two views of 360 held-out images, opened once in T5.

**Fixed settings:** 15 epochs, batch size 64, Adam with learning rate 3e-3, seed 42, cross-entropy loss. The engine enforces them. The **selection metric** is cross-entropy on the *shifted* validation view, which resembles deployment; accuracy is secondary.

In [ ]:
import copy
import math
import os
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.datasets import load_digits
from sklearn.metrics import (ConfusionMatrixDisplay, confusion_matrix,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(2, os.cpu_count() or 1))
SEED = 42
SPLIT_SEED = 2026                    # the same stratified split IDs as Lab 3
EPOCHS, BATCH_SIZE, LR = 15, 64, 3e-3
CANVAS, GLYPH, CENTRE = 32, 16, 8    # a 16x16 digit on a 32x32 field; centred offset = 8
MAX_OFFSET = CANVAS - GLYPH          # 16: furthest top-left position
POSITIVE, MIN_RECALL = 8, 0.90       # the "is it an 8?" detector in T4-T5
PHASES = ['debug', 'arch', 'aug', 'claim', 'extension']
CANDIDATE_PHASES = ['arch', 'aug', 'claim']   # runs that may become the final model
RUNS, TEST_OPENED = {}, False
pd.set_option('display.precision', 4)
pd.set_option('display.width', 160)

print('Python:', platform.python_version(), '| PyTorch:', torch.__version__,
      '| scikit-learn:', sklearn.__version__)
print('Device:', DEVICE, '| CPU threads:', torch.get_num_threads())
print(f'Epochs {EPOCHS} | batch {BATCH_SIZE} | Adam lr {LR} | seed {SEED}')

In [ ]:
#@title Provided data helpers — run this cell; expand to inspect
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def to_glyphs(images8):
    """8x8 scikit-learn digits (values 0-16) -> 16x16 float images in [0, 1]."""
    x = torch.tensor(images8 / 16.0, dtype=torch.float32).unsqueeze(1)
    return F.interpolate(x, size=(GLYPH, GLYPH), mode='bilinear', align_corners=False).clamp(0, 1)

def place(glyphs, rows, cols):
    """Paste each 16x16 glyph onto an empty 32x32 field with its top-left corner at (row, col)."""
    n = glyphs.shape[0]
    field = torch.zeros(n, 1, CANVAS, CANVAS)
    offsets = torch.arange(GLYPH)
    r = (rows.long().view(n, 1) + offsets).view(n, GLYPH, 1)
    c = (cols.long().view(n, 1) + offsets).view(n, 1, GLYPH)
    field[torch.arange(n).view(n, 1, 1), 0, r, c] = glyphs[:, 0]
    return field

def centred(glyphs):
    n = len(glyphs)
    return place(glyphs, torch.full((n,), CENTRE), torch.full((n,), CENTRE))

def fixed_random_positions(n, seed):
    g = torch.Generator().manual_seed(seed)
    return (torch.randint(0, MAX_OFFSET + 1, (n,), generator=g),
            torch.randint(0, MAX_OFFSET + 1, (n,), generator=g))

def normalise(x):
    return (x - DATA['mean']) / DATA['std']

def rotate(glyphs, gen, max_deg):
    n = glyphs.shape[0]
    angle = (torch.rand(n, generator=gen) * 2 - 1) * math.radians(max_deg)
    cos, sin, zero = torch.cos(angle), torch.sin(angle), torch.zeros(n)
    theta = torch.stack([torch.stack([cos, -sin, zero], 1), torch.stack([sin, cos, zero], 1)], 1)
    grid = F.affine_grid(theta, list(glyphs.shape), align_corners=False)
    return F.grid_sample(glyphs, grid, align_corners=False, padding_mode='zeros')

def make_training_batch(glyphs, gen, max_shift=0, rotate_deg=0.0, hflip=False):
    """Training-only augmentation, drawn afresh for every mini-batch. Validation/test are never augmented."""
    n = glyphs.shape[0]
    if rotate_deg > 0:
        glyphs = rotate(glyphs, gen, rotate_deg)
    if hflip:
        flip = torch.rand(n, generator=gen) < 0.5
        glyphs = torch.where(flip.view(n, 1, 1, 1), glyphs.flip(-1), glyphs)
    rows = torch.randint(CENTRE - max_shift, CENTRE + max_shift + 1, (n,), generator=gen)
    cols = torch.randint(CENTRE - max_shift, CENTRE + max_shift + 1, (n,), generator=gen)
    return normalise(place(glyphs, rows, cols))

def prepare_data():
    digits = load_digits()
    ids = np.arange(len(digits.target))
    dev_ids, test_ids = train_test_split(ids, test_size=0.2, stratify=digits.target,
                                         random_state=SPLIT_SEED)
    train_ids, val_ids = train_test_split(dev_ids, test_size=0.25, stratify=digits.target[dev_ids],
                                          random_state=SPLIT_SEED)
    assert set(train_ids).isdisjoint(val_ids) and set(train_ids).isdisjoint(test_ids)
    assert set(val_ids).isdisjoint(test_ids)
    train_g, val_g = to_glyphs(digits.images[train_ids]), to_glyphs(digits.images[val_ids])
    train_canvas = centred(train_g)   # statistics from centred TRAINING images only
    data = dict(train_glyphs=train_g, y_train=torch.tensor(digits.target[train_ids]),
                val_glyphs=val_g, y_val=torch.tensor(digits.target[val_ids]),
                mean=train_canvas.mean().item(), std=train_canvas.std().item(),
                sizes=(len(train_ids), len(val_ids), len(test_ids)))
    rows, cols = fixed_random_positions(len(val_g), seed=202)
    data['val_shift_rows'], data['val_shift_cols'] = rows, cols
    data['val_centred_raw'], data['val_shifted_raw'] = centred(val_g), place(val_g, rows, cols)

    def final_test(run, threshold, positive=POSITIVE):
        """Teaching guard, not security: call once, after the decision is written down."""
        global TEST_OPENED
        if TEST_OPENED:
            raise RuntimeError('The test set was already opened. Reuse final_report; do not retune.')
        if run['phase'] not in CANDIDATE_PHASES or run['status'] != 'complete':
            raise ValueError('Choose a completed T3/T5a candidate run.')
        TEST_OPENED = True
        g, y = to_glyphs(digits.images[test_ids]), torch.tensor(digits.target[test_ids])
        r, c = fixed_random_positions(len(g), seed=303)
        shifted_raw = place(g, r, c)
        probs = predict_proba(run['model'], normalise(shifted_raw))
        return dict(run_name=run['name'], n=len(y), threshold=float(threshold), positive=positive,
                    centred=evaluate(run['model'], normalise(centred(g)), y),
                    shifted=evaluate(run['model'], normalise(shifted_raw), y),
                    y_true=y.numpy(), probs=probs, y_pred=probs.argmax(1),
                    images=shifted_raw[:, 0].numpy())
    return data, final_test

def show_examples():
    fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
    views = [('Training (centred)', centred(DATA['train_glyphs'][:8]), DATA['y_train'][:8]),
             ('Validation (shifted)', DATA['val_shifted_raw'][:8], DATA['y_val'][:8])]
    for row, (title, images, labels) in enumerate(views):
        for i, ax in enumerate(axes[row]):
            ax.imshow(images[i, 0], cmap='gray_r', vmin=0, vmax=1)
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(str(int(labels[i])), fontsize=9)
        axes[row, 0].set_ylabel(title, fontsize=9)
    fig.tight_layout()
    plt.show()

In [ ]:
DATA, open_test_once = prepare_data()
VAL = {'centred': (normalise(DATA['val_centred_raw']), DATA['y_val']),
       'shifted': (normalise(DATA['val_shifted_raw']), DATA['y_val'])}
print('Train / validation / held-out test:', DATA['sizes'])
print('Input shape:', tuple(VAL['shifted'][0].shape), '| labels:', DATA['y_val'].dtype)
print(f"Training-only pixel mean {DATA['mean']:.4f}, std {DATA['std']:.4f}")
print('Validation digits per class:', np.bincount(DATA['y_val'].numpy()))
show_examples()

### Supplied experiment engine

`run_experiment(name, model_fn, phase, max_shift=0, rotate_deg=0, hflip=False)` creates a fresh model with seed 42, trains it for exactly 15 epochs and **keeps the final-epoch model**: no checkpoint selection, so differences come from the model or data, not from stopping. Shuffling and augmentation use separate seeded generators, so two runs see the same mini-batch order even when only one is augmented.

After every epoch the engine records training loss (measured during training, on augmented batches) and cross-entropy/accuracy on both validation views in evaluation mode. Every run is logged with its phase: `debug` (T1), `arch` (T3b), `aug` (T3c), `claim` (T5a), `extension` (optional). Rerunning a name with unchanged settings reuses the result; a name with changed settings raises an error, so poor runs cannot silently disappear from the log.

`max_shift = s` places each training digit at a random top-left position in `[8 − s, 8 + s]` for rows and columns; `s = 8` covers every position. `rotate_deg` rotates by a random angle in `[−rotate_deg, +rotate_deg]`; `hflip` mirrors half the images.

In [ ]:
#@title Provided engine: models, evaluation, training, log
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def backbone():
    """Three blocks of [3x3 conv, padding 1] -> ReLU -> 2x2 max pooling. 32x32 -> 32 x 4 x 4."""
    return [nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)]

def mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(CANVAS * CANVAS, 128), nn.ReLU(), nn.Linear(128, 10))

@torch.no_grad()
def evaluate(model, x, y):
    model.eval()
    logits = model(x.to(DEVICE))
    y = y.to(DEVICE)
    return {'loss': F.cross_entropy(logits, y).item(),
            'acc': (logits.argmax(1) == y).float().mean().item()}

@torch.no_grad()
def predict_proba(model, x):
    model.eval()
    return F.softmax(model(x.to(DEVICE)), dim=1).cpu().numpy()

def run_experiment(name, model_fn, phase, max_shift=0, rotate_deg=0.0, hflip=False,
                   seed=SEED, loss='ce', verbose=True):
    """Fresh model and optimizer; fixed 15-epoch budget; final-epoch model; no test access."""
    if phase not in PHASES:
        raise ValueError(f'phase must be one of {PHASES}')
    if TEST_OPENED and phase != 'extension':
        raise RuntimeError('The test set is open. Only phase="extension" runs are allowed now, '
                           'and they cannot change your T5 decision.')
    if not (0 <= int(max_shift) <= CENTRE and 0 <= float(rotate_deg) <= 180 and loss in ('ce', 'mse')):
        raise ValueError('Use 0 <= max_shift <= 8, 0 <= rotate_deg <= 180, loss "ce" or "mse".')
    aug = dict(max_shift=int(max_shift), rotate_deg=float(rotate_deg), hflip=bool(hflip))
    set_seed(seed)
    model = model_fn().to(DEVICE)
    signature = (str(model), tuple(aug.items()), seed, phase, loss, EPOCHS, LR, BATCH_SIZE)
    if name in RUNS:
        if RUNS[name]['signature'] != signature:
            raise ValueError(f'"{name}" already records a different experiment. Choose a new name; '
                             'the earlier run stays in the log.')
        if verbose:
            print(f'{name}: settings unchanged, reusing the recorded run.')
        return RUNS[name]
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    shuffle_gen = torch.Generator().manual_seed(seed + 1000)
    aug_gen = torch.Generator().manual_seed(seed + 2000)
    x_all, y_all = DATA['train_glyphs'], DATA['y_train']
    history, status, start = [], 'complete', time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = torch.randperm(len(y_all), generator=shuffle_gen)
        total, correct = 0.0, 0
        for s in range(0, len(order), BATCH_SIZE):
            idx = order[s:s + BATCH_SIZE]
            x = make_training_batch(x_all[idx], aug_gen, **aug).to(DEVICE)
            y = y_all[idx].to(DEVICE)
            logits = model(x)
            if loss == 'ce':
                batch_loss = F.cross_entropy(logits, y)
            else:  # regression-style loss on probabilities, used only in deeper investigation D3
                batch_loss = F.mse_loss(F.softmax(logits, 1), F.one_hot(y, 10).float())
            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()
            total += batch_loss.item() * len(idx)
            correct += (logits.argmax(1) == y).sum().item()
        if not math.isfinite(total):
            status = 'failed'
            break
        c, sh = evaluate(model, *VAL['centred']), evaluate(model, *VAL['shifted'])
        history.append(dict(epoch=epoch, train_loss=total / len(y_all), train_acc=correct / len(y_all),
                            val_c_loss=c['loss'], val_c_acc=c['acc'],
                            val_s_loss=sh['loss'], val_s_acc=sh['acc']))
    history = pd.DataFrame(history)
    last = history.iloc[-1] if len(history) else {}
    run = dict(name=name, phase=phase, model_name=model_fn.__name__, model=model.eval(), seed=seed,
               loss=loss, status=status, signature=signature, history=history,
               parameters=count_parameters(model), seconds=time.perf_counter() - start, **aug,
               **{k: float(last[k]) for k in ['train_loss', 'val_c_loss', 'val_c_acc',
                                              'val_s_loss', 'val_s_acc'] if len(history)})
    RUNS[name] = run
    if verbose:
        print(f"{name:22s} {status} | params {run['parameters']:7,d} | {run['seconds']:5.1f} s | "
              f"train loss {run.get('train_loss', float('nan')):.3f} | "
              f"centred val CE {run.get('val_c_loss', float('nan')):.3f} acc {run.get('val_c_acc', float('nan')):.3f} | "
              f"shifted val CE {run.get('val_s_loss', float('nan')):.3f} acc {run.get('val_s_acc', float('nan')):.3f}")
    return run

def experiment_table(phases=None):
    cols = ['name', 'phase', 'model_name', 'max_shift', 'rotate_deg', 'hflip', 'loss', 'seed', 'status',
            'parameters', 'train_loss', 'val_c_loss', 'val_c_acc', 'val_s_loss', 'val_s_acc', 'seconds']
    table = pd.DataFrame([{k: r.get(k) for k in cols} for r in RUNS.values()], columns=cols)
    return table if phases is None else table[table.phase.isin(phases)].reset_index(drop=True)

In [ ]:
#@title Provided analysis and plotting helpers
def plot_curves(runs, title):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for run in runs:
        h = run['history']
        axes[0].plot(h.epoch, h.train_loss, label=run['name'])
        axes[1].plot(h.epoch, h.val_c_acc, label=run['name'])
        axes[2].plot(h.epoch, h.val_s_acc, label=run['name'])
    axes[0].set(xlabel='Epoch', ylabel='Training loss', title='Training loss')
    axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Centred validation', ylim=(0, 1.02))
    axes[2].set(xlabel='Epoch', ylabel='Accuracy', title='Shifted validation', ylim=(0, 1.02))
    for ax in axes:
        ax.grid(alpha=.25)
    axes[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

@torch.no_grad()
def shift_profile(model):
    """Validation accuracy when every digit sits at the vertical centre and column offset 0..16."""
    g, y, n = DATA['val_glyphs'], DATA['y_val'], len(DATA['y_val'])
    return np.array([evaluate(model, normalise(place(g, torch.full((n,), CENTRE), torch.full((n,), col))), y)['acc']
                     for col in range(MAX_OFFSET + 1)])

def plot_shift_profile(runs, title='Accuracy versus horizontal position'):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    for run in runs:
        ax.plot(range(MAX_OFFSET + 1), shift_profile(run['model']), 'o-', ms=3, label=run['name'])
    ax.axvline(CENTRE, color='grey', ls=':', lw=1)
    ax.set(xlabel='Column of the digit\'s left edge (8 = centred, as in training)',
           ylabel='Validation accuracy', ylim=(0, 1.02), xticks=range(0, MAX_OFFSET + 1, 2), title=title)
    ax.grid(alpha=.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()

def show_confusion(model, view='shifted', title=None):
    x, y = VAL[view]
    pred = predict_proba(model, x).argmax(1)
    cm = confusion_matrix(y.numpy(), pred, labels=np.arange(10))
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ConfusionMatrixDisplay(cm, display_labels=np.arange(10)).plot(ax=ax, colorbar=False)
    ax.set(title=title or f'{view} validation | rows = true, columns = predicted')
    fig.tight_layout()
    plt.show()
    return cm

def show_errors(model, label, view='shifted', n=8):
    """Validation errors involving `label` in either direction: true label missed, or wrongly predicted."""
    x, y = VAL[view]
    pred, y = predict_proba(model, x).argmax(1), y.numpy()
    raw = DATA['val_shifted_raw'] if view == 'shifted' else DATA['val_centred_raw']
    idx = np.flatnonzero(((y == label) | (pred == label)) & (pred != y))[:n]
    if len(idx) == 0:
        print(f'No errors involving {label} in the {view} view.')
        return
    fig, axes = plt.subplots(1, len(idx), figsize=(1.6 * len(idx), 2.1), squeeze=False)
    for ax, i in zip(axes.flat, idx):
        ax.imshow(raw[i, 0], cmap='gray_r', vmin=0, vmax=1)
        ax.set_title(f'true {y[i]} / pred {pred[i]}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'Errors involving {label} ({view} validation)', fontsize=10)
    fig.tight_layout()
    plt.show()

def top_confusions(cm, n=4):
    off = [(int(cm[i, j]), i, j) for i in range(len(cm)) for j in range(len(cm)) if i != j and cm[i, j] > 0]
    return ', '.join(f'true {i} -> pred {j}: {c}' for c, i, j in sorted(off, reverse=True)[:n]) or 'none'

def measure_receptive_field(layers, size=64, seed=0):
    """Empirical receptive field: input pixels whose gradient reaches the central output unit."""
    torch.manual_seed(seed)
    net = nn.Sequential(*layers).eval()
    x = torch.randn(16, 1, size, size, requires_grad=True)
    out = net(x)
    out[:, :, out.shape[-2] // 2, out.shape[-1] // 2].sum().backward()
    nz = torch.nonzero(x.grad.abs().sum((0, 1)))
    return int(nz[:, 0].max() - nz[:, 0].min() + 1)

@torch.no_grad()
def equivariance_check(shifts=(3, 4, 8, 16)):
    """Untrained backbone, 64x64 empty field: shift the digits right by s pixels and ask whether the
    final feature map is simply the original map moved by k whole cells (interior cells only)."""
    torch.manual_seed(0)
    net = nn.Sequential(*backbone()).eval()
    g = DATA['val_glyphs'][:64]
    def field(col):
        x = torch.zeros(len(g), 1, 64, 64)
        x[:, :, 24:40, col:col + GLYPH] = g
        return x
    base = net(field(24))
    rows = []
    for s in shifts:
        moved = net(field(24 + s))
        errors = [((moved[..., 1:-1, 1 + k:-1] - base[..., 1:-1, 1:base.shape[-1] - 1 - k]).norm()
                   / base[..., 1:-1, 1:base.shape[-1] - 1 - k].norm()).item() for k in range(3)]
        k = int(np.argmin(errors))
        rows.append(dict(shift_px=s, best_cell_shift=k, relative_error=round(errors[k], 4)))
    return pd.DataFrame(rows)

def check_layer_predictions(predictions, layers):
    rows = []
    for key, (layer, in_shape) in layers.items():
        out = tuple(layer(torch.zeros(1, *in_shape)).shape[1:])
        actual = out + (count_parameters(layer),)
        pred = predictions.get(key)
        rows.append(dict(layer=key, spec=str(layer), input=in_shape, predicted=pred, actual=actual,
                         match='yes' if pred is not None and tuple(pred) == actual else 'NO'))
    return pd.DataFrame(rows)

## T1 · Debug a classification loss · 3 points · 8 minutes

The lecture's two cross-entropy tables (car/rocket/airplane) give the **same accuracy, 2/3**, but different probabilities.

**a.** Compute the average cross-entropy (natural log) of both tables from their probabilities. Also compute what `F.cross_entropy` returns if you pass the probabilities, as if they were logits. Which model is better, and what does accuracy hide? **(1 point)**

**b.** A teammate's CNN "trains, but the loss never drops below about 1.5". Read the code, find the bug, and predict its effect on the loss and on accuracy **before** running. Then train the teammate's model and your fixed version under identical settings. **(1 point)**

**c.** Compute the smallest loss the buggy model can possibly reach with 10 classes. *Hint: its best possible output is a one-hot probability vector, which then passes through softmax again inside the loss.* Compare with the observed loss and explain why accuracy can still improve while the loss is stuck. **(1 point)**

In [ ]:
P1 = torch.tensor([[0.1, 0.1, 0.8], [0.3, 0.4, 0.3], [0.3, 0.3, 0.4]])   # first table: probabilities
P2 = torch.tensor([[0.3, 0.4, 0.3], [0.1, 0.8, 0.1], [0.1, 0.1, 0.8]])   # second table
targets = torch.tensor([0, 1, 2])                                        # car, rocket, airplane

# TODO T1a: average cross-entropy computed correctly from the probabilities.
ce_1 = None
ce_2 = None
# TODO T1a: the value F.cross_entropy returns when the probabilities are passed in as logits.
wrong_1 = None
wrong_2 = None

assert None not in (ce_1, ce_2, wrong_1, wrong_2), 'Complete the four values.'
for name, P, ce, wrong in [('Table 1', P1, ce_1, wrong_1), ('Table 2', P2, ce_2, wrong_2)]:
    acc = (P.argmax(1) == targets).float().mean().item()
    print(f'{name}: accuracy {acc:.3f} | correct CE {float(ce):.4f} | CE with probabilities as logits {float(wrong):.4f}')

### T1a · Your answer

**Correct CE for the two tables, and the values with probabilities as logits:** YOUR VALUES
**Which model is better, and what accuracy hides:** YOUR ANSWER (2 sentences)

In [ ]:
#@title The teammate's model — read it carefully (do not edit)
def teammate_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 4 * 4, 10), nn.Softmax(dim=1))
# The engine trains it with F.cross_entropy(model(x), y), exactly as for every other model.

### T1b · Prediction before running

**The bug and why it matters:** YOUR ANSWER
**Predicted effect on training loss and on accuracy (higher, lower, similar?):** YOUR ANSWER

In [ ]:
def fixed_cnn():
    # TODO T1b: return the teammate's network with the bug removed. Change nothing else.
    raise NotImplementedError('T1b: write fixed_cnn first.')

assert count_parameters(fixed_cnn()) == count_parameters(teammate_cnn()), 'Only remove the bug.'
assert not any(isinstance(m, nn.Softmax) for m in fixed_cnn().modules())
buggy_run = run_experiment('T1_teammate', teammate_cnn, 'debug')
fixed_run = run_experiment('T1_fixed', fixed_cnn, 'debug')
display(experiment_table(['debug'])[['name', 'parameters', 'train_loss', 'val_c_loss', 'val_c_acc']])
plot_curves([buggy_run, fixed_run], 'T1: identical settings, one module removed')

In [ ]:
# TODO T1c: the smallest possible loss of the buggy model for 10 classes (use math.log, math.e).
loss_floor = None

assert loss_floor is not None, 'Compute the floor.'
print(f'Theoretical floor: {loss_floor:.4f}')
print(f"Teammate model: lowest epoch training loss {buggy_run['history'].train_loss.min():.4f}, "
      f"final centred validation accuracy {buggy_run['val_c_acc']:.3f}")
print(f"Fixed model:    lowest epoch training loss {fixed_run['history'].train_loss.min():.4f}, "
      f"final centred validation accuracy {fixed_run['val_c_acc']:.3f}")

### T1b–c · Results and interpretation

**Observed training loss and centred validation accuracy, buggy vs fixed:** YOUR VALUES
**Was your prediction right? Explain the floor and why accuracy can still rise:** YOUR ANSWER (3 sentences)

## T2 · Predict shapes, parameters and receptive fields · 4 points · 10 minutes

For a convolution or pooling layer with kernel $k$, stride $s$, padding $p$ and dilation $d$:

$$H_\text{out} = \left\lfloor \frac{H + 2p - d(k-1) - 1}{s} \right\rfloor + 1, \qquad \text{conv parameters} = C_\text{out}\,(C_\text{in}\, k^2 + 1).$$

**a.** Fill in `PREDICTIONS` with `(C_out, H_out, W_out, parameters)` for layers A–G **without running them**, then run the checker. For every `NO`, explain the mistake. State which settings change the parameter count and which change only the output size. **(2 points)**

**b.** Predict the receptive field (in pixels, one side of the square) of one output unit of three stacks: S1, three 3×3 convolutions; S2, the same with dilations 1, 2, 4; S3, the backbone used in T3. Then measure. Does one final S3 unit see a whole 16-pixel digit? **(1 point)**

**c.** The S3 backbone has a total stride of 8. Predict what happens to its final feature map when the input digit moves right by 3, 4, 8 and 16 pixels. Which shifts give the same map, just moved by whole cells? Then run the provided check on an **untrained** backbone. Why can untrained weights test this? **(1 point)**

In [ ]:
LAYERS = {   # name: (layer, input shape C x H x W)
    'A': (nn.Conv2d(1, 8, kernel_size=3), (1, 32, 32)),
    'B': (nn.Conv2d(1, 8, kernel_size=3, stride=2, padding=1), (1, 32, 32)),
    'C': (nn.Conv2d(1, 8, kernel_size=3, padding=2, dilation=2), (1, 32, 32)),
    'D': (nn.Conv2d(3, 12, kernel_size=5), (3, 32, 32)),
    'E': (nn.MaxPool2d(2), (16, 32, 32)),
    'F': (nn.Conv2d(16, 32, kernel_size=3, stride=3), (16, 32, 32)),
    'G': (nn.Conv2d(32, 32, kernel_size=1), (32, 4, 4)),
}
# TODO T2a: replace each None by your prediction (C_out, H_out, W_out, parameters) BEFORE running.
PREDICTIONS = {'A': None, 'B': None, 'C': None, 'D': None, 'E': None, 'F': None, 'G': None}

assert all(v is not None for v in PREDICTIONS.values()), 'T2a: predict all seven layers first.'
display(check_layer_predictions(PREDICTIONS, LAYERS))

### T2a · Your answer

**Mismatches and the reason for each (or "none"):** YOUR ANSWER
**Settings that change the parameter count vs only the output size:** YOUR ANSWER (2 sentences)

### T2b–c · Predictions before running

**Receptive fields S1 / S2 / S3, with your reasoning:** YOUR VALUES
**Shifts of 3, 4, 8, 16 px: which give the same feature map moved by whole cells?** YOUR ANSWER

In [ ]:
STACKS = {
    'S1: 3 x conv3x3': [nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(), nn.Conv2d(4, 4, 3, padding=1), nn.ReLU(),
                        nn.Conv2d(4, 4, 3, padding=1)],
    'S2: dilations 1, 2, 4': [nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(),
                              nn.Conv2d(4, 4, 3, padding=2, dilation=2), nn.ReLU(),
                              nn.Conv2d(4, 4, 3, padding=4, dilation=4)],
    'S3: T3 backbone': backbone(),
}
RF_PREDICTIONS = {'S1: 3 x conv3x3': None, 'S2: dilations 1, 2, 4': None, 'S3: T3 backbone': None}  # TODO T2b

assert all(v is not None for v in RF_PREDICTIONS.values()), 'T2b: predict the three receptive fields.'
display(pd.DataFrame([dict(stack=k, predicted=RF_PREDICTIONS[k], measured=measure_receptive_field(v),
                           parameters=count_parameters(nn.Sequential(*v))) for k, v in STACKS.items()]))
display(equivariance_check())

### T2b–c · Results and interpretation

**Measured receptive fields; does one S3 unit see a whole digit?** YOUR ANSWER
**Equivariance result, why untrained weights suffice, and what this predicts for T3:** YOUR ANSWER (3 sentences)

## T3 · Does a CNN handle digits it has only seen centred? · 6 points · 17 minutes

Convolution is *equivariant*: move the input and the feature map moves too. A classifier needs *invariance*: the same answer wherever the digit is. The **head** decides how the feature map becomes 10 logits:

| Model | Head after the S3 backbone | Idea |
| --- | --- | --- |
| `mlp` (provided) | none: 1,024 pixels → 128 → 10 | baseline from earlier labs |
| `cnn_flatten` | flatten the 32×4×4 map → linear → 10 | keeps *where* each feature is |
| `cnn_gap` | global **average** pooling → linear → 10 | averages each map over all positions |
| `cnn_gmp` | global **max** pooling → linear → 10 | keeps each map's strongest response |

**a.** Before training, predict the ranking of the four models on **centred** and on **shifted** validation accuracy, using T2c. **(1 point)**

**b.** Complete `make_cnn`, then train all four models on centred training data (phase `arch`). Describe the table, the curves and the position profile. Which predictions failed, and why? **(2 points)**

**c.** Now switch on training-time shift augmentation for **one** architecture of your choice (phase `aug`). Choose `max_shift` (1–8) and predict the result first. You may add a second augmented run if you justify it. Compare with the *same* architecture without augmentation and interpret: whether convolution alone delivered invariance, whether the MLP-versus-CNN comparison is fair given their parameter counts, and one limitation of this evidence. **(3 points)**

### T3a · Prediction before running

**Predicted ranking on centred / shifted validation accuracy, with one reason:** YOUR ANSWER

In [ ]:
def make_cnn(head):
    """The S3 backbone (output 32 x 4 x 4) followed by one of three heads, ending in 10 logits."""
    layers = backbone()               # built first, so the seed gives the same backbone weights as T1
    if head == 'flatten':
        flat_features = None          # TODO T3b: channels x height x width after the backbone
        head_layers = None if flat_features is None else [nn.Flatten(), nn.Linear(flat_features, 10)]
    elif head == 'gap':
        head_layers = None            # TODO T3b: global average pooling to 1x1, flatten, linear -> 10
    elif head == 'gmp':
        head_layers = None            # TODO T3b: global max pooling to 1x1, flatten, linear -> 10
    else:
        raise ValueError(head)
    if head_layers is None:
        raise NotImplementedError(f'T3b: complete the "{head}" head.')
    return nn.Sequential(*layers, *head_layers)

def cnn_flatten(): return make_cnn('flatten')
def cnn_gap(): return make_cnn('gap')
def cnn_gmp(): return make_cnn('gmp')

for fn in [mlp, cnn_flatten, cnn_gap, cnn_gmp]:
    out = fn()(torch.zeros(2, 1, CANVAS, CANVAS))
    assert out.shape == (2, 10), f'{fn.__name__} must return 10 logits per image.'
    print(f'{fn.__name__:12s} {count_parameters(fn()):8,d} trainable parameters')

In [ ]:
arch_runs = [run_experiment(f'T3_{fn.__name__}', fn, 'arch') for fn in [mlp, cnn_flatten, cnn_gap, cnn_gmp]]
display(experiment_table(['arch'])[['name', 'parameters', 'train_loss', 'val_c_loss', 'val_c_acc',
                                    'val_s_loss', 'val_s_acc', 'seconds']])
plot_curves(arch_runs, 'T3b: trained on centred digits only')
plot_shift_profile(arch_runs, 'T3b: validation accuracy vs horizontal position (vertical position centred)')

### T3b · Observations

**Centred and shifted validation accuracy/CE of the four models:** YOUR VALUES
**Which predictions failed, and why? Use the profile plot and T2c:** YOUR ANSWER (3–4 sentences)

### T3c · Plan and prediction before running

**Architecture, `max_shift`, and why:** YOUR ANSWER
**Predicted centred / shifted validation accuracy compared with the same model unaugmented:** YOUR ANSWER

In [ ]:
# TODO T3c: one augmented run (a second is optional, if justified in your plan).
AUG_PLAN = [
    dict(name='T3_aug_1', model_fn=None, max_shift=None),
    # dict(name='T3_aug_2', model_fn=..., max_shift=...),   # optional second run
]

assert 1 <= len(AUG_PLAN) <= 2
assert all(p['model_fn'] in (mlp, cnn_flatten, cnn_gap, cnn_gmp) and isinstance(p['max_shift'], int)
           and 1 <= p['max_shift'] <= 8 for p in AUG_PLAN), 'Choose a model function and 1 <= max_shift <= 8.'
aug_runs = [run_experiment(p['name'], p['model_fn'], 'aug', max_shift=p['max_shift']) for p in AUG_PLAN]
paired = [RUNS[f"T3_{p['model_fn'].__name__}"] for p in AUG_PLAN]
display(experiment_table(['arch', 'aug'])[['name', 'model_name', 'max_shift', 'parameters', 'train_loss',
                                           'val_c_loss', 'val_c_acc', 'val_s_loss', 'val_s_acc']])
plot_curves(paired + aug_runs, 'T3c: training curves with and without augmentation')
plot_shift_profile(paired + aug_runs, 'T3c: same architecture with and without shift augmentation')

### T3c · Interpretation

**Paired result (same architecture, with vs without augmentation) and whether your prediction held:** YOUR VALUES AND ANSWER
**Did convolution alone deliver invariance? Is MLP vs CNN a fair comparison here?** YOUR ANSWER (3 sentences)
**One limitation of this evidence:** YOUR ANSWER

## T4 · Evaluate beyond accuracy · 4 points · 12 minutes

The provided cell picks the current best T3 run by **shifted-validation CE** (ties: fewer parameters). Everything in T4 uses the shifted validation view.

**a.** Complete `one_vs_rest(cm, k)` for class `k` of a confusion matrix whose rows are true classes. First check it on the lecture's rocket/car/airplane matrix with *rocket* as the positive class: the four counts must add up to 84. Then apply it to digit 8 and compare with scikit-learn. Which digit is most often confused with 8, in either direction (an 8 missed, or another digit called 8), and why might that be? **(2 points)**

**b.** The operator wants a detector "is this an 8?" that catches **at least 90% of 8s** (recall ≥ 0.90). Complete `choose_threshold` to return the threshold on p(8) with the highest precision among those meeting the recall target (ties: the higher threshold). Compare it with the default arg-max decision. **(1 point)**

**c.** Compute accuracy, recall and F1 of a "never predict 8" rule. Explain why accuracy misleads here, and why the threshold must be chosen on validation rather than test. **(1 point)**

In [ ]:
def current_best(phases):
    table = experiment_table(phases).query("status == 'complete'")
    return RUNS[table.sort_values(['val_s_loss', 'parameters', 'name']).iloc[0]['name']]

t4_run = current_best(['arch', 'aug'])
print(f"T4 model: {t4_run['name']} | shifted val CE {t4_run['val_s_loss']:.4f}, accuracy {t4_run['val_s_acc']:.3f}")
cm_val = show_confusion(t4_run['model'], 'shifted', f"{t4_run['name']} | shifted validation")
VAL_PROBS = predict_proba(t4_run['model'], VAL['shifted'][0])
Y_VAL = DATA['y_val'].numpy()

In [ ]:
def one_vs_rest(cm, k):
    """Treat class k as positive and all other classes as negative. Rows = true, columns = predicted."""
    cm = np.asarray(cm)
    TP = None   # TODO T4a
    FP = None   # TODO T4a
    FN = None   # TODO T4a
    TN = None   # TODO T4a
    if None in (TP, FP, FN, TN):
        raise NotImplementedError('T4a: complete the four counts.')
    precision = TP / (TP + FP) if TP + FP else 0.0
    recall = TP / (TP + FN) if TP + FN else 0.0
    return dict(TP=int(TP), FP=int(FP), FN=int(FN), TN=int(TN), precision=precision, recall=recall,
                specificity=TN / (TN + FP), FPR=FP / (FP + TN),
                F1=2 * precision * recall / (precision + recall) if precision + recall else 0.0)

LECTURE_CM = np.array([[15, 3, 8], [4, 23, 1], [9, 2, 19]])   # rows: rocket, car, airplane (true)
rocket = one_vs_rest(LECTURE_CM, 0)
print('Lecture matrix, rocket = positive:', {k: (v if isinstance(v, int) else round(float(v), 3)) for k, v in rocket.items()})
assert rocket['TP'] + rocket['FP'] + rocket['FN'] + rocket['TN'] == LECTURE_CM.sum() == 84

eight = one_vs_rest(cm_val, POSITIVE)
p, r, f1, _ = precision_recall_fscore_support(Y_VAL, VAL_PROBS.argmax(1), labels=[POSITIVE], zero_division=0)
assert np.allclose([eight['precision'], eight['recall'], eight['F1']], [p[0], r[0], f1[0]])
print('Digit 8, shifted validation:', {k: (v if isinstance(v, int) else round(float(v), 3)) for k, v in eight.items()})
print('Row of true 8s      (predicted 0..9):', cm_val[POSITIVE])
print('Column of predicted 8s (true 0..9):  ', cm_val[:, POSITIVE])
show_errors(t4_run['model'], POSITIVE)

### T4a · Your answer

**Rocket counts TP/FP/FN/TN; digit-8 precision, recall, F1:** YOUR VALUES
**Most frequent confusion involving 8 (either direction) and a plausible reason from the images:** YOUR ANSWER (2 sentences)

In [ ]:
def choose_threshold(y_binary, scores, min_recall=MIN_RECALL):
    """Highest-precision threshold t (predict positive when score >= t) with recall >= min_recall.
    Ties: the higher threshold. Use only validation data."""
    best = None
    for t in np.unique(scores):
        pred = scores >= t
        # TODO T4b: compute recall and precision of `pred`; keep t if it meets the recall target and
        # beats the current best precision (ties: higher t). Store best = (t, precision, recall).
        pass
    if best is None:
        raise NotImplementedError('T4b: complete choose_threshold.')
    return best

y8_val = (Y_VAL == POSITIVE)
t_star, prec_star, rec_star = choose_threshold(y8_val, VAL_PROBS[:, POSITIVE])
fpr, tpr, _ = roc_curve(y8_val, VAL_PROBS[:, POSITIVE])
auc = roc_auc_score(y8_val, VAL_PROBS[:, POSITIVE])
print(f'ROC AUC for "is it an 8?" on shifted validation: {auc:.4f}')
argmax_pred = VAL_PROBS.argmax(1) == POSITIVE
chosen_pred = VAL_PROBS[:, POSITIVE] >= t_star
for label, pred in [('arg-max decision', argmax_pred), (f'threshold {t_star:.4f}', chosen_pred)]:
    m = one_vs_rest(confusion_matrix(y8_val, pred, labels=[False, True]), 1)
    print(f"{label:22s} recall {m['recall']:.3f} | precision {m['precision']:.3f} | FPR {m['FPR']:.4f} | F1 {m['F1']:.3f}")
fig, ax = plt.subplots(figsize=(4.4, 4))
ax.plot(fpr, tpr, label=f'ROC, AUC = {auc:.4f}')
ax.plot([0, 1], [0, 1], ':', color='grey', label='random classifier')
m = one_vs_rest(confusion_matrix(y8_val, chosen_pred, labels=[False, True]), 1)
ax.scatter([m['FPR']], [m['recall']], color='red', zorder=3, label='chosen threshold')
ax.set(xlabel='False positive rate', ylabel='True positive rate (recall)', title='"Is it an 8?" (shifted validation)')
ax.legend(fontsize=8); ax.grid(alpha=.25); fig.tight_layout(); plt.show()

In [ ]:
# TODO T4c: the "never predict 8" rule on shifted validation. Build its 2x2 confusion matrix with
# confusion_matrix(y8_val, never_pred, labels=[False, True]) and reuse one_vs_rest(..., 1).
never_pred = None
never = None

assert never is not None, 'T4c: evaluate the never-8 rule.'
never_acc = (never['TP'] + never['TN']) / len(y8_val)
model_m = one_vs_rest(confusion_matrix(y8_val, chosen_pred, labels=[False, True]), 1)
model_acc = (model_m['TP'] + model_m['TN']) / len(y8_val)
print(f"Never-8 rule : accuracy {never_acc:.3f} | recall {never['recall']:.3f} | F1 {never['F1']:.3f}")
print(f"Thresholded  : accuracy {model_acc:.3f} | recall {model_m['recall']:.3f} | F1 {model_m['F1']:.3f}")

### T4b–c · Your answer

**Chosen threshold, its validation recall/precision, the AUC, and the comparison with arg-max:** YOUR VALUES
**Why accuracy misleads for the never-8 rule:** YOUR ANSWER
**Why the threshold must come from validation, not test:** YOUR ANSWER

## T5 · Test a claim, freeze the decision, test once · 3 points · 8 minutes

**a. Claim check (at most two runs, phase `claim`).** Choose one path. **(1 point, including the declaration at the top)**

- **Path A — you used an AI assistant.** Ask it for one change that should improve shifted-validation performance under this engine's fixed budget (augmentation settings or a new `model_fn`). Summarise the suggestion, predict its effect, test it, and say whether the evidence supports it.
- **Path B — supplied claim:** *"Stronger augmentation is always better: add random rotations of up to ±180° and horizontal flips to the shift augmentation."* Test it against your T3c augmented run with the same architecture. Use the second run, if needed, to separate possible explanations.

**b. Freeze.** The provided rule selects the candidate (T3 or T5a) with the lowest shifted-validation CE (ties: fewer parameters) and recomputes the 8-threshold on validation. You may override the rule only with a written, validation-based reason. Write the decision **before** opening the test. **(1 point)**

**c. Test once.** Report centred and shifted test accuracy/CE and the 8-detector's recall and precision at your threshold. Compare cautiously with validation. Do not change anything afterwards. **(1 point)**

### T5a · Claim and prediction before running

**Path (A or B); the claim or AI suggestion in one sentence:** YOUR ANSWER
**Prediction, and what result would contradict the claim:** YOUR ANSWER

In [ ]:
CLAIM_PATH = None      # TODO T5a: 'A' or 'B'
CLAIM_BASE = None      # TODO T5a: name of the comparison run, e.g. your T3c augmented run
CLAIM_PLAN = [         # TODO T5a: one or two runs. Example of the supplied claim (Path B):
    # dict(name='T5_claim_1', model_fn=cnn_gmp, max_shift=8, rotate_deg=180, hflip=True),
]

assert CLAIM_PATH in ('A', 'B') and CLAIM_BASE in RUNS and 1 <= len(CLAIM_PLAN) <= 2, 'Complete the T5a plan.'
claim_runs = [run_experiment(p.pop('name'), p.pop('model_fn'), 'claim', **p) for p in copy.deepcopy(CLAIM_PLAN)]
display(experiment_table(['aug', 'claim'])[['name', 'model_name', 'max_shift', 'rotate_deg', 'hflip',
                                            'train_loss', 'val_c_acc', 'val_s_loss', 'val_s_acc']])
for run in [RUNS[CLAIM_BASE]] + claim_runs:
    cm = show_confusion(run['model'], 'shifted', f"{run['name']} | shifted validation")
    print(f"{run['name']}: largest confusions | {top_confusions(cm)}")

### T5a · Result

**Validation result versus the comparison run:** YOUR VALUES
**Does the evidence support the claim? Explain any failure using the confusion matrices and training loss:** YOUR ANSWER (3 sentences)

In [ ]:
candidates = experiment_table(CANDIDATE_PHASES).query("status == 'complete'")
display(candidates.sort_values(['val_s_loss', 'parameters'])[
    ['name', 'phase', 'model_name', 'max_shift', 'rotate_deg', 'hflip', 'parameters', 'val_s_loss', 'val_s_acc', 'val_c_acc']])
rule_choice = current_best(CANDIDATE_PHASES)['name']
FINAL_OVERRIDE = None   # Optional: a candidate name, only with a written validation-based reason in T5b.
final_run = RUNS[FINAL_OVERRIDE or rule_choice]
final_probs_val = predict_proba(final_run['model'], VAL['shifted'][0])
FINAL_THRESHOLD = choose_threshold(y8_val, final_probs_val[:, POSITIVE])[0]
print(f"Rule choice: {rule_choice} | final run: {final_run['name']} | "
      f"shifted val CE {final_run['val_s_loss']:.4f}, acc {final_run['val_s_acc']:.3f} | "
      f"8-threshold from validation: {FINAL_THRESHOLD:.4f}")

### T5b · Decision before opening the test

**Final run, its validation evidence, the threshold, and one limitation:** YOUR ANSWER (3–4 sentences)

In [ ]:
DECISION_WRITTEN = False   # TODO T5b: set True only after writing the decision above.
assert DECISION_WRITTEN, 'Write and freeze the decision before opening the test.'
if 'final_report' not in globals():
    final_report = open_test_once(final_run, FINAL_THRESHOLD)
r = final_report
test8 = one_vs_rest(confusion_matrix(r['y_true'] == POSITIVE, r['probs'][:, POSITIVE] >= r['threshold'],
                                     labels=[False, True]), 1)
print(f"Test n = {r['n']} | model {r['run_name']}")
print(f"Centred test: CE {r['centred']['loss']:.4f}, accuracy {r['centred']['acc']:.3f}")
print(f"Shifted test: CE {r['shifted']['loss']:.4f}, accuracy {r['shifted']['acc']:.3f}")
print(f"8-detector at {r['threshold']:.4f}: recall {test8['recall']:.3f} ({test8['TP']}/{test8['TP'] + test8['FN']}), "
      f"precision {test8['precision']:.3f}, FPR {test8['FPR']:.4f}")
print(f"Validation for comparison: shifted CE {final_run['val_s_loss']:.4f}, accuracy {final_run['val_s_acc']:.3f}")
cm_test = confusion_matrix(r['y_true'], r['y_pred'], labels=np.arange(10))
fig, ax = plt.subplots(figsize=(5.2, 4.6))
ConfusionMatrixDisplay(cm_test, display_labels=np.arange(10)).plot(ax=ax, colorbar=False)
ax.set_title('Held-out shifted test | rows = true, columns = predicted'); fig.tight_layout(); plt.show()

### T5c · Final interpretation

**Test results (centred, shifted, 8-detector) and the comparison with validation:** YOUR VALUES
**What the validation–test difference can and cannot tell you:** YOUR ANSWER (2 sentences)

In [ ]:
# Required final evidence: keep this table in the submitted notebook.
final_log = experiment_table()
display(final_log.drop(columns=['loss']).round(4))
print('Recorded runs:', len(final_log), '| total training time:', round(final_log.seconds.sum(), 1), 's')
print('Final model:', final_report['run_name'], '| test opened once:', TEST_OPENED)

## Check and submit · final 5 minutes

1. Fill in your name, Neptun code and assistance declaration. Replace every `YOUR ANSWER`/`YOUR VALUES` and complete all TODOs.
2. Keep all outputs visible: tables, curves, position profiles, confusion matrices, the ROC plot, the final log and your written predictions. Unsuccessful runs are evidence; do not delete them.
3. Check the notebook from top to bottom. If you restart and rerun, reproduce the **same frozen plan**; do not change it after seeing the test results.
4. Rename the notebook **`YOUR_NEPTUN_DL_Lab04.ipynb`**, replacing `YOUR_NEPTUN` with your Neptun code, then choose **File → Download → Download .ipynb**.
5. Upload the executed notebook to **Moodle by the end of the lab** and confirm the submission. A Colab link alone is not sufficient.

If you cannot finish, submit your partial work on time and state what is missing. If you do not have Moodle access, keep the downloaded file, tell the instructor before the deadline and email the work as described in the course README. Keep completed answers and personal details out of the public repository.

## Deeper investigations · optional, after the core

These are for students who finish early or want to go further. Use phase `extension`. **Results here cannot change your T5 decision**, because the test set is already open. For each investigation you attempt, write a prediction first and a 2–3 sentence conclusion.

- **D1 · Seed robustness.** Retrain your final configuration and the runner-up with seeds 7 and 17 (`seed=` argument). Does the T5 ranking survive? What does that say about margins smaller than the seed-to-seed spread?
- **D2 · Pooling aliasing.** The position profiles of `T3_cnn_gap` and `T3_cnn_gmp` repeat with different periods. Measure the period of each and explain it from the strides in T2. *Hint: what does a global max over a 4×4 map that was itself max-pooled from 8×8 equal?*
- **D3 · Regression-style loss.** The lecture contrasts MSE for regression with softmax + cross-entropy for classification. Train your final configuration with `loss='mse'` (MSE between softmax probabilities and one-hot targets). Compare learning speed and shifted-validation CE, and explain the difference through the gradients.
- **D4 · Dilation instead of pooling.** Build a network without pooling whose dilated 3×3 convolutions reach a receptive field of at least 22 pixels, followed by global max pooling. Compare parameters, training time and shifted accuracy with your final model. When is the trade-off worth it?

In [ ]:
# Optional: your deeper-investigation code. Example:
# d1 = [run_experiment(f'D1_final_s{s}', cnn_gmp, 'extension', max_shift=8, seed=s) for s in (7, 17)]

### Deeper investigations · your notes (optional)

YOUR ANSWER

## References and credits

- BME lecture: *Deep Learning — Classification & Convolutional Neural Networks* (20 October 2026), supplied course slides.
- [scikit-learn: the 8×8 digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html), derived from the UCI Optical Recognition of Handwritten Digits data (E. Alpaydin, C. Kaynak). This is **not MNIST**. The 32×32 placement is an artificial classroom transformation, not a new benchmark.
- [scikit-learn: metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) and [Google ML Crash Course: ROC and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc).
- [PyTorch: `torch.nn.Conv2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) (output-size formula), [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) (expects unnormalised logits) and [`AdaptiveMaxPool2d`](https://docs.pytorch.org/docs/stable/generated/torch.nn.AdaptiveMaxPool2d.html).
- Dumoulin & Visin (2016), [A guide to convolution arithmetic for deep learning](https://arxiv.org/abs/1603.07285).
- Zhang (2019), [Making convolutional networks shift-invariant again](https://arxiv.org/abs/1904.11486), on aliasing caused by strided pooling.

Course material: Dr. Mohammed Salah Al-Radhi, BME. Dataset and framework credits remain with their authors.